# Viewing the AXI4-Stream Timing

In this demo, you will see:

- How to generate a VCD file with traces of signals
- Load the VCD and parse the VCD file in python with Waveflow's `VcdParser` utility
- Extract signals for **AXI4-Stream interface**
- Extract individual AXI4-Stream **bursts** from the parsed stream
- Deserialize the data in the burst into Waveflow data schemas and arrays for analysis
- Plot the bursts with a Waveflow **TimingDiagram**

## Pre-Requisites

Prior to running this notebook, you should run the demo:

```bash
python -m examples.stream_inband.poly_build --through validate_timing
```

This command will run the C-simulation, C synthesis and RTL co-simulation.  The RTL co-simulation is run with tracing so that the signals are captured for viewing.  We also import the following packages.

In [ ]:
from waveflow.hw.arrayutils import read_array
from poly import Float32, PolyCmdHdr, PolyRespHdr


## Generating a VCD file

The Vitis C simulation does not produce a timing waveform.  The build's `error_vcd` step co-simulates the `early_tlast_vcd` scenario with port tracing and converts the trace to a VCD:

```bash
python poly_build.py --through error_vcd
```

That writes `vcd/error_path.vcd`, which is committed, so you can run this notebook without Vitis.  (By hand, the conversion is `xsim_vcd --top poly --comp vcd_proj --out error_path.vcd --trace_level port`.)  We load the VCD with the `vcdvcd` package:

In [ ]:
import os
from vcdvcd import VCDVCD

vcd_file = 'error_path.vcd'
fn = os.path.join(os.getcwd(), 'vcd', vcd_file)
if not os.path.exists(fn):
    raise FileNotFoundError(f"VCD file not found: {fn}")

# Parse the VCD file
vcd = VCDVCD(fn, signals=None, store_tvs=True)


It is useful to display the signals.  In this case, there are a large number of signals since we have captured every signal in the simulation.  

We also print the signals with the name `TDATA`. 

In [ ]:
nsig = len(vcd.signals)
print(f"Number of signals in VCD: {nsig}")

# Find the signals with TDATA and TREADY in their names
tdatas = [s for s in vcd.signals if 'TDATA' in s]
print(tdatas)


In [ ]:
from waveflow.utils.vcd import VcdParser
from waveflow.utils.timing import TimingDiagram

## Extracing the Signals for the AXI4-Stream

The above print out of the signals shows two relevant signals for the AXI4-Stream:

- `apatb_poly_top.AESL_inst_poly.s_in_TDATA[31:0]`
- `apatb_poly_top.AESL_inst_poly.m_out_TDATA[31:0]`

The signals are the data signals for the input and output streams of the Vitis kernel `poly`.  We can extract these signals and the clock as follows:


In [ ]:
# Create a parsing class
vp = VcdParser(vcd)

# Get the clock signal name
clk_name = vp.add_clock_signal()

top_name = 'AESL_inst_poly'
s_in_name = f"{top_name}.s_in_"
m_out_name = f"{top_name}.m_out_"

# Get the  AXI-Stream command signals
in_str_sigs, in_bw = vp.add_axiss_signals(name=s_in_name, short_name_prefix='s_in',
                                           ignore_multiple=True)
print(in_str_sigs)

# Get the output AXI-Stream signals
out_str_sigs, out_bw = vp.add_axiss_signals(name=m_out_name, short_name_prefix='m_out',
                                            ignore_multiple=True)
print(out_str_sigs)


## Creating a Timing Diagram

We can now create a timing diagram from the parsed signals as follows.  

In [ ]:
# Get the timing signals
sig_list = vp.get_td_signals()

# Create the timing diagram
td = TimingDiagram()
td.add_signals(sig_list)
trange = None
ax = td.plot_signals(add_clk_grid=True, trange=trange, 
                text_scale_factor=1e4, text_mode='never')
_ = ax.set_xlabel('Time [ns]')

## Getting the Burst Timing and Deserializing the Burst Data

The VCD parser has a method `extract_axis_bursts` that will extract the bursts in the stream.  We use this to print the bursts.  The run sends two `DATA` commands, so the input has four bursts: a command header, its samples, the second header, and its samples -- which end after 6 of the 10 announced, with an early TLAST.  The output has four bursts: a response header and the results for each command.

In [ ]:
# Extract the AXI-Stream bursts and print the burst information for the input
bursts_in, clk_period= vp.extract_axis_bursts(clk_name, in_str_sigs)
print('Input AXI-Stream bursts:')
for i, burst in enumerate(bursts_in):
    nbeats = len(burst['beat_type'])
    nbeats_transfer = sum(1 for bt in burst['beat_type'] if bt == 0)
    print(f"Burst {i}: tstart = {burst['tstart']}, bt={burst['beat_type']}, nbeats_transfer = {nbeats_transfer}")

# Extract the AXI-Stream output bursts
print('\nOutput AXI-Stream bursts:')
bursts_out, clk_period= vp.extract_axis_bursts(clk_name, out_str_sigs)
for i, burst in enumerate(bursts_out):
    nbeats = len(burst['beat_type'])
    nbeats_transfer = sum(1 for bt in burst['beat_type'] if bt == 0)
    print(f"Burst {i}: tstart = {burst['tstart']}, bt={burst['beat_type']}, nbeats_transfer = {nbeats_transfer}")


We can also extract the values by deserializing the captured information on the bursts.

In [ ]:
word_bw = 32
for k in range(2):                      # the two DATA commands, in order
    cmd_hdr = PolyCmdHdr()
    cmd_hdr.deserialize(word_bw=word_bw, packed=bursts_in[2 * k]['data'])
    print("PolyCmdHdr values:")
    for f, v in cmd_hdr.val.items():
        print(f"    {f}: {v}")
    # The burst may end early: decode the samples actually sent.
    nsamp = int(cmd_hdr.val['nsamp'])
    nsent = min(nsamp, len(bursts_in[2 * k + 1]['data']))
    x = read_array(packed=bursts_in[2 * k + 1]['data'], word_bw=word_bw, elem_type=Float32,
                   shape=(nsent,))
    print(f"  x ({nsent} of {nsamp} sent): {x.val}")

    resp_hdr = PolyRespHdr()
    resp_hdr.deserialize(word_bw=word_bw, packed=bursts_out[2 * k]['data'])
    y = read_array(packed=bursts_out[2 * k + 1]['data'], word_bw=word_bw, elem_type=Float32,
                   shape=(nsent,))
    print(f"  response tx_id = {resp_hdr.val['tx_id']}, y: {y.val}")


Finally, we replot the timing diagram with the bursts color coded so that you can see the timing.

In [ ]:
from matplotlib.patches import Patch

# Create the timing diagram
td = TimingDiagram()
td.add_signals(sig_list)
trange = None
ax = td.plot_signals(add_clk_grid=True, trange=trange, 
                text_scale_factor=1e4, text_mode='never')
_ = ax.set_xlabel('Time [ns]')

def color_burst(sig_name, burst, color, clk_period):
    t0 = burst['tstart']
    t1 = t0 + len(burst['beat_type']) * clk_period
    td.add_patch(sig_name=sig_name, time=[t0, t1], color=color, alpha=0.3)

hdr_color = 'orange'
data_color = 'green'
# Headers are the even bursts, data the odd ones, on both streams.
for k, b in enumerate(bursts_in):
    color_burst('s_in_TDATA', b, hdr_color if k % 2 == 0 else data_color, clk_period)
for k, b in enumerate(bursts_out):
    color_burst('m_out_TDATA', b, hdr_color if k % 2 == 0 else data_color, clk_period)

legend_elements = [
    Patch(facecolor='orange', edgecolor='black', alpha=0.3, label='header'),
    Patch(facecolor='green', edgecolor='black', alpha=0.3, label='data'),
]
ax.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1.02, 0.5), borderaxespad=0.)

